### Tasks

* 1. Train a simple neural network to classify whether a review is positive or negative using a small dataset (you can use any dataset of your choice), then save the trained model to a .h5 file using model.save('sentiment_model.h5').

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, TextVectorization, Embedding, GlobalAveragePooling1D, Flatten

In [2]:
train_reviews = np.array([
    "Great movie, absolute masterpiece!",
    "Terrible acting, complete waste of time.",
    "Loved the cinematography and performance.",
    "Boring storyline, fell asleep halfway.",
    "Brilliant direction and fantastic score.",
    "Horrible dialogue and cheap effects."
], dtype=object)

In [3]:
train_labels = np.array([1, 0, 1, 0, 1, 0])    # 1 = Positive, 0 = Negative

In [4]:
vectorizer = TextVectorization(max_tokens=100, output_sequence_length=10)
vectorizer.adapt(train_reviews)

In [5]:
model_1 = Sequential([
    Input(shape=(1,), dtype=tf.string),
    vectorizer,
    Embedding(input_dim=100, output_dim=16),
    GlobalAveragePooling1D(),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [6]:
model_1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [7]:
model_1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [8]:
model_1.fit(train_reviews, train_labels, epochs=20, verbose=0)

In [9]:
model_1.save('sentiment_model.h5')
print("Model saved successfully to 'sentiment_model.h5'.")

Model saved successfully to 'sentiment_model.h5'.


In [10]:
model_1.save('sentiment_model.keras')

* 2. Write code to load the 'sentiment_model.h5' file you saved and use it to predict the sentiment of three new sample reviews.

In [11]:
from tensorflow.keras.models import load_model

In [12]:
loaded_model = load_model('sentiment_model.keras')

C:\Users\patel prit\AppData\Roaming\Python\Python312\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 7 variables whereas the saved optimizer has 12 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [13]:
new_reviews = np.array([
    "An absolute cinematic triumph with amazing acting!",
    "The plot was terrible and incredibly boring.",
    "Not bad, pretty decent execution and good pacing."
], dtype=object)

In [14]:
predictions = loaded_model.predict(new_reviews, verbose=0)

In [15]:
for review, prob in zip(new_reviews, predictions.flatten()):
    sentiment = "Positive" if prob >= 0.5 else "Negative"
    print(f"Review: {review}")
    print(f"Predicted Score: {prob:.4f} = Sentiment: {sentiment}\n")

Review: An absolute cinematic triumph with amazing acting!
Predicted Score: 0.4977 = Sentiment: Negative

Review: The plot was terrible and incredibly boring.
Predicted Score: 0.5035 = Sentiment: Positive

Review: Not bad, pretty decent execution and good pacing.
Predicted Score: 0.5058 = Sentiment: Positive



* 3. Implement model checkpointing during training so that the model's weights are saved every time the validation accuracy improves.<br><br><em><strong>Hint:</strong> Use the ModelCheckpoint callback in Keras with save_best_only=True.</em>

In [16]:
from tensorflow.keras.callbacks import ModelCheckpoint

In [17]:
checkpoint_cb = ModelCheckpoint(
    filepath='best_sentiment_model.keras',
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

model_1.fit(
    train_reviews[:4], train_labels[:4],
    validation_data=(train_reviews[4:], train_labels[4:]),
    epochs=10,
    callbacks=[checkpoint_cb],
    verbose=0
)


Epoch 1: val_accuracy improved from None to 1.00000, saving model to best_sentiment_model.keras

Epoch 1: finished saving model to best_sentiment_model.keras

Epoch 2: val_accuracy did not improve from 1.00000

Epoch 3: val_accuracy did not improve from 1.00000

Epoch 4: val_accuracy did not improve from 1.00000

Epoch 5: val_accuracy did not improve from 1.00000

Epoch 6: val_accuracy did not improve from 1.00000

Epoch 7: val_accuracy did not improve from 1.00000

Epoch 8: val_accuracy did not improve from 1.00000

Epoch 9: val_accuracy did not improve from 1.00000

Epoch 10: val_accuracy did not improve from 1.00000


* 4. Export your trained model in a format suitable for deployment (such as TensorFlow SavedModel format), and explain in one line how this format helps with deploying the model to a web or mobile app.

In [18]:
model_1.export('saved_model_export')
print("Complete: Exported to 'saved_model_export' folder.")

INFO:tensorflow:Assets written to: saved_model_export\assets


INFO:tensorflow:Assets written to: saved_model_export\assets


Saved artifact at 'saved_model_export'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.string, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2882861595920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2882861595344: TensorSpec(shape=(), dtype=tf.int64, name=None)
  2882861590352: TensorSpec(shape=(), dtype=tf.string, name=None)
  2882861591888: TensorSpec(shape=(), dtype=tf.int64, name=None)
  2882861592080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2882861594576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2882861593424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2882861594768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2882861594192: TensorSpec(shape=(), dtype=tf.resource, name=None)
Complete: Exported to 'saved_model_export' folder.


**Why SavedModel for Deployment:**

* The SavedModel format encapsulates the model's graph, weights, and pre-processing pipeline into a standalone, language-agnostic directory—allowing seamless serving via TensorFlow Serving (REST/gRPC APIs), TensorFlow Lite (mobile/edge devices), or TensorFlow.js (web browsers) without needing the original Python training code.

* 5. Use ChatGPT or Copilot to help you write the code for saving both the model architecture and weights separately, then test loading them back and confirm the loaded model gives the same predictions as the original.

In [19]:
model_config = model_1.get_config()

model_1.save_weights("model_weights.weights.h5")
print("* Architecture (Config) and Weights saved separately.")

reconstructed_model = Sequential.from_config(model_config)

reconstructed_model.layers[0].set_vocabulary(model_1.layers[0].get_vocabulary())

reconstructed_model.load_weights("model_weights.weights.h5")

test_samples = tf.convert_to_tensor(["Truly incredible experience!", "Awful film, hated it."], dtype=tf.string)

original_preds = model_1.predict(test_samples, verbose=0)
reconstructed_preds = reconstructed_model.predict(test_samples, verbose=0)

print("\nPrediction Verification:")
print("Original Model Outputs= ", original_preds.flatten())
print("Reconstructed Model Outputs=", reconstructed_preds.flatten())

np.testing.assert_allclose(original_preds, reconstructed_preds, rtol=1e-5)
print("\nSUCCESS: Loaded model predictions perfectly match the original model")

* Architecture (Config) and Weights saved separately.

Prediction Verification:
Original Model Outputs=  [0.5090739  0.50777686]
Reconstructed Model Outputs= [0.5090739  0.50777686]

SUCCESS: Loaded model predictions perfectly match the original model
